## BME 4810 — Neural Network Models
#### Bhakti Moradiya & Srihitha Mitta

Three neural network classifiers on the OCT-MNIST dataset:

| Model | Architecture | Input |
|-------|-------------|-------|
| **CNN** | 2-block Conv + Dense head | 28×28×1 image |
| **MLP** | 4-layer Dense (improved) | F_rich 998-dim features |
| **RNN** | Bidirectional LSTM | 28 row-sequences × 28 |


---
### Step 1 — Load Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from scipy.ndimage import sobel
from skimage.measure import block_reduce

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, precision_score,
                              recall_score, f1_score)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, regularizers

print(f'TensorFlow : {tf.__version__}')
print(f'Keras      : {keras.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPU        : {len(gpus) > 0}  {gpus}')


---
### Step 2 — Load OCT-MNIST Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

list_files = np.load('/content/drive/MyDrive/octmnist.npz')

train_images = list_files['train_images'].astype(np.float32) / 255.0
train_labels = list_files['train_labels'].flatten()
val_images   = list_files['val_images'].astype(np.float32)   / 255.0
val_labels   = list_files['val_labels'].flatten()
test_images  = list_files['test_images'].astype(np.float32)  / 255.0
test_labels  = list_files['test_labels'].flatten()

CLASS_NAMES = {0: 'Normal', 1: 'Choroidal Neovascularization',
               2: 'Diabetic Macular Edema', 3: 'Drusen'}
CLASS_SHORT  = ['Normal', 'CNV', 'DME', 'Drusen']

print(f'Train : {train_images.shape}  |  Val : {val_images.shape}  |  Test : {test_images.shape}')


---
### Step 3 — Feature Helper Functions (from M1/M2)

In [ ]:
def apply_sobel_subsample(images, target_size=(14, 14)):
    result = []
    for img in images:
        sx  = sobel(img, axis=0); sy = sobel(img, axis=1)
        edges = np.hypot(sx, sy); edges /= (edges.max() + 1e-8)
        block = (img.shape[0]//target_size[0], img.shape[1]//target_size[1])
        result.append(block_reduce(edges, block_size=block, func=np.max))
    return np.array(result)

def get_statistical_features(images):
    feats = []
    for img in images:
        flat = img.flatten()
        feats.append([
            flat.mean(), flat.std(), flat.min(), flat.max(),
            np.percentile(flat, 25), np.percentile(flat, 50),
            np.percentile(flat, 75), np.percentile(flat, 90),
            img[:7,:7].mean(), img[:7,7:].mean(),
            img[7:,:7].mean(), img[7:,7:].mean(),
            np.diff(img, axis=0).std(), np.diff(img, axis=1).std(),
            (flat > flat.mean()).sum()/len(flat),
            np.sum(flat**2), np.sum(flat*np.log(flat+1e-8)), flat.var(),
        ])
    return np.array(feats)

print('Helper functions ready.')


---
### Step 4 — Balanced Dataset & F_rich Features

In [ ]:
np.random.seed(42)
N_PER_CLASS = 2436
bal_idx = []
for cls in range(4):
    idx = np.where(train_labels == cls)[0]
    bal_idx.extend(np.random.choice(idx, N_PER_CLASS, replace=False))
bal_idx = np.array(bal_idx)
X_bal   = train_images[bal_idx]
Y_bal   = train_labels[bal_idx].astype('int32')
print(f'Balanced dataset: {X_bal.shape}  ({N_PER_CLASS} per class)')

# ── F_rich: raw pixels + Sobel features + statistical features (998 dims)
X_bal_sobel = apply_sobel_subsample(X_bal)
F_rich = np.hstack([
    X_bal.reshape(len(X_bal), -1),             # 784
    X_bal_sobel.reshape(len(X_bal_sobel), -1), # 196
    get_statistical_features(X_bal_sobel)      # 18
])  # → 998

scaler_mlp = StandardScaler()
F_rich_sc  = scaler_mlp.fit_transform(F_rich)

# Test features
X_test_sobel = apply_sobel_subsample(test_images)
F_test_rich  = np.hstack([
    test_images.reshape(len(test_images), -1),
    X_test_sobel.reshape(len(X_test_sobel), -1),
    get_statistical_features(X_test_sobel)
])
F_test_rich_sc = scaler_mlp.transform(F_test_rich)

print(f'F_rich_sc      : {F_rich_sc.shape}')
print(f'F_test_rich_sc : {F_test_rich_sc.shape}')


---
---
## Model 1 — Convolutional Neural Network (CNN)

| Layer | Type | Output |
|-------|------|--------|
| Input | — | 28×28×1 |
| Conv2D(32,3×3) + BN + ReLU | Block 1 | 28×28×32 |
| Conv2D(32,3×3) + BN + ReLU | Block 1 | 28×28×32 |
| MaxPool(2×2) | Block 1 | 14×14×32 |
| Conv2D(64,3×3) + BN + ReLU | Block 2 | 14×14×64 |
| Conv2D(64,3×3) + BN + ReLU | Block 2 | 14×14×64 |
| MaxPool(2×2) | Block 2 | 7×7×64 |
| Flatten → Dense(256) + Dropout(0.5) | Head | 256 |
| Dense(4, softmax) | Output | 4 |


#### CNN — Prepare Inputs

In [ ]:
X_cnn = X_bal[:, :, :, np.newaxis].astype('float32')
Y_cnn = Y_bal

X_tr, X_vl, Y_tr, Y_vl = train_test_split(
    X_cnn, Y_cnn, test_size=0.2, stratify=Y_cnn, random_state=42)

X_te = test_images[:, :, :, np.newaxis].astype('float32')
Y_te = test_labels.astype('int32')

print(f'Train {X_tr.shape} | Val {X_vl.shape} | Test {X_te.shape}')


#### CNN — Build & Train

In [ ]:
def build_cnn(input_shape=(28, 28, 1), n_classes=4, dropout=0.5):
    model = keras.Sequential([
        keras.Input(shape=input_shape),
        # Block 1
        layers.Conv2D(32, (3,3), padding='same'),
        layers.BatchNormalization(), layers.Activation('relu'),
        layers.Conv2D(32, (3,3), padding='same'),
        layers.BatchNormalization(), layers.Activation('relu'),
        layers.MaxPooling2D((2,2)),
        # Block 2
        layers.Conv2D(64, (3,3), padding='same'),
        layers.BatchNormalization(), layers.Activation('relu'),
        layers.Conv2D(64, (3,3), padding='same'),
        layers.BatchNormalization(), layers.Activation('relu'),
        layers.MaxPooling2D((2,2)),
        # Head
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(dropout),
        layers.Dense(n_classes, activation='softmax'),
    ], name='OCT_CNN')
    return model

cnn_model = build_cnn()
cnn_model.summary()


In [ ]:
cnn_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cnn_cb = [
    callbacks.EarlyStopping(monitor='val_accuracy', patience=10,
                            restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                patience=5, min_lr=1e-6, verbose=1),
]

history_cnn = cnn_model.fit(
    X_tr, Y_tr,
    validation_data=(X_vl, Y_vl),
    epochs=50, batch_size=64,
    callbacks=cnn_cb, verbose=1
)


#### CNN — Evaluate

In [ ]:
test_loss_cnn, acc_cnn = cnn_model.evaluate(X_te, Y_te, verbose=0)
Y_pred_cnn = np.argmax(cnn_model.predict(X_te, verbose=0), axis=1)

print(f'CNN  Test Accuracy : {acc_cnn:.4f}')
print(f'CNN  Test Loss     : {test_loss_cnn:.4f}')
print()
print(classification_report(Y_te, Y_pred_cnn, target_names=CLASS_SHORT))


In [ ]:
# Training curves
ep = range(1, len(history_cnn.history['loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('CNN Training Curves (OCT-MNIST, Balanced, n=9,744)', fontsize=12, fontweight='bold')

axes[0].plot(ep, history_cnn.history['loss'],         'b-o', ms=3, label='Train')
axes[0].plot(ep, history_cnn.history['val_loss'],     'r-o', ms=3, label='Val')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(ep, [a*100 for a in history_cnn.history['accuracy']],     'b-o', ms=3, label='Train')
axes[1].plot(ep, [a*100 for a in history_cnn.history['val_accuracy']], 'r-o', ms=3, label='Val')
axes[1].set_title('Accuracy (%)'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('fig_nn_cnn_training.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Confusion matrix — row-normalized
cm_cnn      = confusion_matrix(Y_te, Y_pred_cnn)
cm_cnn_norm = cm_cnn.astype('float') / cm_cnn.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm_cnn_norm, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_xticklabels(CLASS_SHORT, rotation=30, ha='right')
ax.set_yticklabels(CLASS_SHORT)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'CNN Confusion Matrix (acc={acc_cnn:.4f})')
for i in range(4):
    for j in range(4):
        ax.text(j, i, f'{cm_cnn_norm[i,j]:.2f}', ha='center', va='center',
                fontsize=9, color='white' if cm_cnn_norm[i,j]>0.5 else 'black')
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig('fig_nn_cnn_confusion.png', dpi=150, bbox_inches='tight')
plt.show()


---
---
## Model 2 — Multilayer Perceptron (MLP) — Improved

**Improvements over original M3 MLP:**
- **Input:** F_rich (998-dim: pixels + Sobel + stats) instead of raw pixels only — richer signal
- **Architecture:** Proper BN placement (Dense → BN → Activation → Dropout) at every layer
- **Regularization:** L2 weight decay on all Dense layers + Dropout at every hidden layer
- **Training:** Longer patience (15 epochs), cosine-style LR decay with `ReduceLROnPlateau`
- **Batch size:** 128 (larger mini-batches give more stable gradient estimates)

| Layer | Output | Dropout |
|-------|--------|---------|
| Dense(512) + BN + ReLU | 512 | 0.4 |
| Dense(256) + BN + ReLU | 256 | 0.4 |
| Dense(128) + BN + ReLU | 128 | 0.3 |
| Dense(64)  + BN + ReLU | 64  | 0.2 |
| Dense(4, softmax) | 4 | — |


#### MLP — Prepare Inputs (F_rich features)

In [ ]:
# Use F_rich_sc (998 engineered features) — far more informative than raw pixels
X_mlp = F_rich_sc.astype('float32')
Y_mlp = Y_bal

X_tr_mlp, X_vl_mlp, Y_tr_mlp, Y_vl_mlp = train_test_split(
    X_mlp, Y_mlp, test_size=0.2, stratify=Y_mlp, random_state=42)

X_te_mlp  = F_test_rich_sc.astype('float32')
Y_te_mlp  = test_labels.astype('int32')

print(f'MLP Train : {X_tr_mlp.shape}  |  Val : {X_vl_mlp.shape}  |  Test : {X_te_mlp.shape}')


#### MLP — Build & Train

In [ ]:
def build_mlp_improved(input_dim=998, n_classes=4):
    reg = regularizers.l2(1e-4)
    model = keras.Sequential([
        keras.Input(shape=(input_dim,)),

        layers.Dense(512, kernel_regularizer=reg),
        layers.BatchNormalization(), layers.Activation('relu'),
        layers.Dropout(0.4),

        layers.Dense(256, kernel_regularizer=reg),
        layers.BatchNormalization(), layers.Activation('relu'),
        layers.Dropout(0.4),

        layers.Dense(128, kernel_regularizer=reg),
        layers.BatchNormalization(), layers.Activation('relu'),
        layers.Dropout(0.3),

        layers.Dense(64, kernel_regularizer=reg),
        layers.BatchNormalization(), layers.Activation('relu'),
        layers.Dropout(0.2),

        layers.Dense(n_classes, activation='softmax'),
    ], name='OCT_MLP_Improved')
    return model

mlp_model = build_mlp_improved(input_dim=X_tr_mlp.shape[1])
mlp_model.summary()


In [ ]:
mlp_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

mlp_cb = [
    callbacks.EarlyStopping(monitor='val_loss', patience=15,
                            restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                patience=6, min_lr=1e-6, verbose=1),
]

history_mlp = mlp_model.fit(
    X_tr_mlp, Y_tr_mlp,
    validation_data=(X_vl_mlp, Y_vl_mlp),
    epochs=80, batch_size=128,
    callbacks=mlp_cb, verbose=1
)


#### MLP — Evaluate

In [ ]:
test_loss_mlp, acc_mlp = mlp_model.evaluate(X_te_mlp, Y_te_mlp, verbose=0)
Y_pred_mlp = np.argmax(mlp_model.predict(X_te_mlp, verbose=0), axis=1)

print(f'MLP  Test Accuracy : {acc_mlp:.4f}')
print(f'MLP  Test Loss     : {test_loss_mlp:.4f}')
print()
print(classification_report(Y_te_mlp, Y_pred_mlp, target_names=CLASS_SHORT))


In [ ]:
# Training curves
ep = range(1, len(history_mlp.history['loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('MLP (Improved) Training Curves — Input: F_rich 998-dim', fontsize=12, fontweight='bold')

axes[0].plot(ep, history_mlp.history['loss'],         'b-o', ms=3, label='Train')
axes[0].plot(ep, history_mlp.history['val_loss'],     'r-o', ms=3, label='Val')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(ep, [a*100 for a in history_mlp.history['accuracy']],     'b-o', ms=3, label='Train')
axes[1].plot(ep, [a*100 for a in history_mlp.history['val_accuracy']], 'r-o', ms=3, label='Val')
axes[1].set_title('Accuracy (%)'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('fig_nn_mlp_training.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Confusion matrix
cm_mlp      = confusion_matrix(Y_te_mlp, Y_pred_mlp)
cm_mlp_norm = cm_mlp.astype('float') / cm_mlp.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm_mlp_norm, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_xticklabels(CLASS_SHORT, rotation=30, ha='right')
ax.set_yticklabels(CLASS_SHORT)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'MLP (Improved) Confusion Matrix (acc={acc_mlp:.4f})')
for i in range(4):
    for j in range(4):
        ax.text(j, i, f'{cm_mlp_norm[i,j]:.2f}', ha='center', va='center',
                fontsize=9, color='white' if cm_mlp_norm[i,j]>0.5 else 'black')
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig('fig_nn_mlp_confusion.png', dpi=150, bbox_inches='tight')
plt.show()


---
---
## Model 3 — Recurrent Neural Network (Bidirectional LSTM)

Each 28×28 OCT image is treated as a **sequence of 28 rows** (time steps),  
where each row is a 28-dimensional feature vector.  
The BiLSTM reads rows both top→bottom and bottom→top, capturing how retinal  
layers transition spatially — a structure well-suited to OCT pathology.

| Layer | Output | Notes |
|-------|--------|-------|
| Input | (28, 28) | 28 time steps × 28 features |
| Bidirectional LSTM(64) | 128 | forward + backward pass |
| Dropout(0.4) | 128 | — |
| Dense(128) + BN + ReLU | 128 | representation head |
| Dropout(0.3) | 128 | — |
| Dense(4, softmax) | 4 | class probabilities |


#### RNN — Prepare Inputs

In [ ]:
# (N, 28, 28) — sequence of 28 rows, each row = 28 pixel values
X_rnn = X_bal.astype('float32')
Y_rnn = Y_bal

X_tr_rnn, X_vl_rnn, Y_tr_rnn, Y_vl_rnn = train_test_split(
    X_rnn, Y_rnn, test_size=0.2, stratify=Y_rnn, random_state=42)

X_te_rnn = test_images.astype('float32')
Y_te_rnn = test_labels.astype('int32')

print(f'RNN Train : {X_tr_rnn.shape}  ({X_tr_rnn.shape[1]} time steps × {X_tr_rnn.shape[2]} features)')
print(f'RNN Val   : {X_vl_rnn.shape}')
print(f'RNN Test  : {X_te_rnn.shape}')


#### RNN — Build & Train

In [ ]:
def build_rnn(time_steps=28, n_features=28, n_classes=4):
    model = keras.Sequential([
        keras.Input(shape=(time_steps, n_features)),
        layers.Bidirectional(layers.LSTM(64, return_sequences=False)),
        layers.Dropout(0.4),
        layers.Dense(128),
        layers.BatchNormalization(), layers.Activation('relu'),
        layers.Dropout(0.3),
        layers.Dense(n_classes, activation='softmax'),
    ], name='OCT_RNN_BiLSTM')
    return model

rnn_model = build_rnn()
rnn_model.summary()


In [ ]:
rnn_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

rnn_cb = [
    callbacks.EarlyStopping(monitor='val_loss', patience=10,
                            restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                patience=5, min_lr=1e-6, verbose=1),
]

history_rnn = rnn_model.fit(
    X_tr_rnn, Y_tr_rnn,
    validation_data=(X_vl_rnn, Y_vl_rnn),
    epochs=60, batch_size=64,
    callbacks=rnn_cb, verbose=1
)


#### RNN — Evaluate

In [ ]:
test_loss_rnn, acc_rnn = rnn_model.evaluate(X_te_rnn, Y_te_rnn, verbose=0)
Y_pred_rnn = np.argmax(rnn_model.predict(X_te_rnn, verbose=0), axis=1)

print(f'RNN  Test Accuracy : {acc_rnn:.4f}')
print(f'RNN  Test Loss     : {test_loss_rnn:.4f}')
print()
print(classification_report(Y_te_rnn, Y_pred_rnn, target_names=CLASS_SHORT))


In [ ]:
# Training curves
ep = range(1, len(history_rnn.history['loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('RNN (BiLSTM) Training Curves — OCT-MNIST Balanced', fontsize=12, fontweight='bold')

axes[0].plot(ep, history_rnn.history['loss'],         'b-o', ms=3, label='Train')
axes[0].plot(ep, history_rnn.history['val_loss'],     'r-o', ms=3, label='Val')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(ep, [a*100 for a in history_rnn.history['accuracy']],     'b-o', ms=3, label='Train')
axes[1].plot(ep, [a*100 for a in history_rnn.history['val_accuracy']], 'r-o', ms=3, label='Val')
axes[1].set_title('Accuracy (%)'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('fig_nn_rnn_training.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Confusion matrix
cm_rnn      = confusion_matrix(Y_te_rnn, Y_pred_rnn)
cm_rnn_norm = cm_rnn.astype('float') / cm_rnn.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm_rnn_norm, cmap='Purples', vmin=0, vmax=1)
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_xticklabels(CLASS_SHORT, rotation=30, ha='right')
ax.set_yticklabels(CLASS_SHORT)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'RNN (BiLSTM) Confusion Matrix (acc={acc_rnn:.4f})')
for i in range(4):
    for j in range(4):
        ax.text(j, i, f'{cm_rnn_norm[i,j]:.2f}', ha='center', va='center',
                fontsize=9, color='white' if cm_rnn_norm[i,j]>0.5 else 'black')
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig('fig_nn_rnn_confusion.png', dpi=150, bbox_inches='tight')
plt.show()


---
### All 3 Models — Training Curves Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Neural Network Models — Training Comparison (OCT-MNIST)', fontsize=12, fontweight='bold')

for hist, label, ls in [
    (history_cnn, 'CNN',        '-'),
    (history_mlp, 'MLP (Impr)', '--'),
    (history_rnn, 'RNN BiLSTM', ':'),
]:
    ep = range(1, len(hist.history['loss']) + 1)
    axes[0].plot(ep, [a*100 for a in hist.history['accuracy']],     ls, label=f'{label} Train')
    axes[0].plot(ep, [a*100 for a in hist.history['val_accuracy']], ls+'', label=f'{label} Val', alpha=0.7)
    axes[1].plot(ep, hist.history['loss'],     ls, label=f'{label} Train')
    axes[1].plot(ep, hist.history['val_loss'], ls+'', label=f'{label} Val', alpha=0.7)

axes[0].set_title('Accuracy (%)'); axes[0].set_xlabel('Epoch')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)
axes[1].set_title('Loss'); axes[1].set_xlabel('Epoch')
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('fig_nn_all_training_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


---
### Final Summary — Neural Network Models

In [ ]:
print('=' * 58)
print(f'  {"MODEL":<28}  {"INPUT":<15}  {"TEST ACC":>8}')
print('-' * 58)
for name, inp, acc in [
    ('CNN',             '28×28×1 image',    acc_cnn),
    ('MLP (Improved)',  'F_rich 998-dim',   acc_mlp),
    ('RNN (BiLSTM)',    '28×28 sequences',  acc_rnn),
]:
    print(f'  {name:<28}  {inp:<15}  {acc:>8.4f}')
print('=' * 58)
best = max(acc_cnn, acc_mlp, acc_rnn)
print(f'  Best: {best:.4f}')
